# Japanese SRT Transcription with Kotoba-Whisper and Qwen Forced Aligner

This notebook transcribes Japanese video or audio into a Japanese `.srt` subtitle file.

- ASR: `kotoba-tech/kotoba-whisper-v2.2-faster`
- Forced alignment: `Qwen/Qwen3-ForcedAligner-0.6B`

The notebook clones this repository automatically, so it can be opened from GitHub and run directly in Colab.


In [ ]:
!nvidia-smi
!python --version


## Install System Tools


In [ ]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg git


## Clone Project Repository


In [ ]:
from pathlib import Path
import sys

REPO_DIR = Path("/content/KotobaSub")
if not REPO_DIR.exists():
    !git clone --depth 1 https://github.com/BilyHurington/KotobaSub.git /content/KotobaSub

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(f"Project repo: {REPO_DIR}")


## Install Python Dependencies


In [ ]:
!pip install -q -r /content/KotobaSub/requirements-colab.txt


## Configure Workspace and Import Helpers


In [ ]:
from src.config import MODEL_CONFIG, SUBTITLE_CONFIG, WORKSPACE_CONFIG
from src.audio import ensure_directories, extract_audio_16k_mono, audio_output_path
from src.transcribe import load_kotoba_model, transcribe_audio, build_alignment_text
from src.align import load_qwen_aligner, run_qwen_alignment
from src.subtitles import build_subtitles_from_units, write_srt

WORK_DIR = WORKSPACE_CONFIG.root
INPUT_DIR = WORKSPACE_CONFIG.input_dir
AUDIO_DIR = WORKSPACE_CONFIG.audio_dir
OUTPUT_DIR = WORKSPACE_CONFIG.output_dir

ensure_directories(WORKSPACE_CONFIG.all_dirs())

LANGUAGE = MODEL_CONFIG.language
USE_VAD = MODEL_CONFIG.use_vad
BEAM_SIZE = MODEL_CONFIG.beam_size
WHISPER_COMPUTE_TYPES = MODEL_CONFIG.whisper_compute_types
KOTOBA_MODEL_ID = MODEL_CONFIG.kotoba_model_id
QWEN_ALIGNER_MODEL_ID = MODEL_CONFIG.qwen_aligner_model_id
FALLBACK_TO_WHISPER_TIMESTAMPS = MODEL_CONFIG.fallback_to_whisper_timestamps


## Upload Input Media


In [ ]:
from google.colab import files
import shutil

uploaded = files.upload()

input_paths = []
for name in uploaded.keys():
    src = Path(name)
    dst = INPUT_DIR / src.name
    shutil.move(str(src), str(dst))
    input_paths.append(dst)

if not input_paths:
    raise RuntimeError("No input file was uploaded.")

input_path = input_paths[0]
print(f"Input: {input_path}")


## Extract Audio


In [ ]:
audio_path = audio_output_path(input_path, AUDIO_DIR)
extract_audio_16k_mono(input_path, audio_path)
print(f"Audio: {audio_path}")


## Transcribe with Kotoba-Whisper


In [ ]:
whisper_model, whisper_compute_type = load_kotoba_model(
    KOTOBA_MODEL_ID,
    WHISPER_COMPUTE_TYPES,
)

whisper_segments, info = transcribe_audio(
    whisper_model,
    audio_path,
    language=LANGUAGE,
    beam_size=BEAM_SIZE,
    use_vad=USE_VAD,
)

alignment_text = build_alignment_text(whisper_segments)
print(f"Detected language: {info.language} ({info.language_probability:.2f})")
print(f"Whisper compute type: {whisper_compute_type}")
print(f"Whisper segments: {len(whisper_segments)}")
print(alignment_text[:1000])


## Clone Qwen3-ASR and Align


In [ ]:
from pathlib import Path
import sys

QWEN_REPO_DIR = Path("/content/Qwen3-ASR")
if not QWEN_REPO_DIR.exists():
    !git clone --depth 1 https://github.com/QwenLM/Qwen3-ASR.git /content/Qwen3-ASR

if str(QWEN_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(QWEN_REPO_DIR))

aligner = load_qwen_aligner(QWEN_ALIGNER_MODEL_ID)


In [ ]:
try:
    aligned_units = run_qwen_alignment(
        audio_path=audio_path,
        transcript_text=alignment_text,
        aligner=aligner,
    )
    used_alignment = "qwen"
except Exception as exc:
    if not FALLBACK_TO_WHISPER_TIMESTAMPS:
        raise

    print("Qwen alignment failed. Falling back to Whisper timestamps.")
    print(repr(exc))
    aligned_units = whisper_segments
    used_alignment = "whisper"

print(f"Alignment source: {used_alignment}")
print(f"Aligned units: {len(aligned_units)}")


## Build SRT Subtitles


In [ ]:
subtitle_segments = build_subtitles_from_units(aligned_units, config=SUBTITLE_CONFIG)

srt_path = OUTPUT_DIR / f"{input_path.stem}.ja.srt"
write_srt(subtitle_segments, srt_path)

print(f"Subtitle segments: {len(subtitle_segments)}")
print(f"SRT: {srt_path}")


## Download SRT


In [ ]:
from google.colab import files

files.download(str(srt_path))
